In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, jaccard_score
from scipy.stats import zscore
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
data = pd.read_csv('/content/drive/MyDrive/creditcard.csv')
df=pd.DataFrame(data)
df.head()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0.0
1,0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0.0
2,1,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0.0
3,1,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0.0
4,2,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0.0


In [ ]:

df.dropna(axis=0,inplace=True)
print(df.isnull().sum())
X = df.drop(columns=['Class'])  # Features
y = df['Class']  # Labels (used for Jaccard evaluation)
# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, stratify=y, random_state=42)

print("Training set shape: ", X_train.shape)
print("Test set shape: ", X_test.shape)

Time      0
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       0
V15       0
V16       0
V17       0
V18       0
V19       0
V20       0
V21       0
V22       0
V23       0
V24       0
V25       0
V26       0
V27       0
V28       0
Amount    0
Class     0
dtype: int64
Training set shape:  (13927, 30)
Test set shape:  (5970, 30)


In [ ]:
# K-means
kmeans = KMeans(n_clusters=2, random_state=42)
kmeans_labels = kmeans.fit_predict(X_train)

# Hierarchical Clustering
hierarchical = AgglomerativeClustering(n_clusters=2)
hierarchical_labels = hierarchical.fit_predict(X_train)


# DBSCAN
dbscan = DBSCAN(eps=0.5, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_train)


print("Results Before Removing Outliers:")
print("kmean label",kmeans_labels)
print("hierarchical label",hierarchical_labels)
print("dbscan label",dbscan_labels)
print("kmeans silhoutte score",silhouette_score(X_train, kmeans_labels))
print("hierarchical silhoutte score",silhouette_score(X_train, hierarchical_labels))
print("dbscan silhoutte score",silhouette_score(X_train, dbscan_labels))



Results Before Removing Outliers:
kmean label [0 0 0 ... 0 0 0]
hierarchical label [0 0 0 ... 0 0 0]
dbscan label [ 0 -1 -1 ... -1 -1 -1]
kmeans silhoutte score 0.791569509511617
hierarchical silhoutte score 0.8056599747269375
dbscan silhoutte score -0.3711599817417082


In [ ]:
from scipy.stats import zscore

# Calculate Z-scores
z_scores = np.abs(zscore(X_train))
outliers = (z_scores > 3).any(axis=1)

# Remove Outliers
X_train_clean = X_train[~outliers]
y_train_clean = y_train[~outliers]

print(f"Original training set size: {X_train.shape[0]}")
print(f"Cleaned training set size: {X_train_clean.shape[0]}")

Original training set size: 13927
Cleaned training set size: 12334


In [ ]:
# K-means
kmeans_clean = KMeans(n_clusters=2, random_state=42)
kmeans_clean_labels = kmeans_clean.fit_predict(X_train_clean)
kmeans_clean_silscore=silhouette_score(X_train_clean, kmeans_clean_labels)

# Hierarchical Clustering
hierarchical_clean = AgglomerativeClustering(n_clusters=2)
hierarchical_clean_labels = hierarchical_clean.fit_predict(X_train_clean)
hierarchical_clean_silscore=silhouette_score(X_train_clean, hierarchical_clean_labels)

# DBSCAN
dbscan_clean = DBSCAN(eps=0.5, min_samples=5)
dbscan_clean_labels = dbscan_clean.fit_predict(X_train_clean)
dbscan_clean_silscore=silhouette_score(X_train_clean, dbscan_clean_labels)

print("\nResults After Removing Outliers:")
print("kmeans_clean_results",kmeans_clean_silscore)
print("hierarchical_clean_results",hierarchical_clean_silscore)
print("dbscan_clean_results",dbscan_clean_silscore)




Results After Removing Outliers:
kmeans_clean_results 0.13387642201327146
hierarchical_clean_results 0.0733661732029224
dbscan_clean_results -0.34169791959007734
